In [ ]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"PyTorch version: {torch.__version__}")
# print(f"GPU device: {torch.cuda.get_device_name(0)}")
print(f"Number of GPUs: {torch.cuda.device_count()}")

CUDA available: True
CUDA version: 12.1
PyTorch version: 2.5.1+cu121
GPU device: NVIDIA GeForce RTX 4060 Laptop GPU
Number of GPUs: 1


In [ ]:
import logging

from dotenv import load_dotenv
from langchain_core.documents import Document

from rag.ingestion.llama_parse_processor import process_document
from rag.ingestion.document_fetcher import DocumentFetcher
from rag.ingestion.document_tracker import DocumentTracker
from rag.ingestion.vector_store import QdrantManager
from utils.api_utils import DefineEdgeFundamentalsAPI
from utils.data_helpers import (
    initialize_metadata_data,
    initialize_stock_data,
    symbol_to_fincode,
)

load_dotenv()

logger = logging.getLogger(__name__)


await initialize_stock_data()
await initialize_metadata_data()

### Get Stocks from Nifty 50 Index

In [ ]:
api = DefineEdgeFundamentalsAPI()

groups = await api.get_predefined_groups()

all_group_names = groups.keys()
for gname in all_group_names:
    if gname == "Nifty 50 Index":
        nifty_50_stocks = groups[gname]

symbol_to_fincode_map = {}
for symbol in nifty_50_stocks:
    symbol_to_fincode_map[symbol] = symbol_to_fincode(symbol)

fincodes = list(symbol_to_fincode_map.values())

In [ ]:
import os

from joblib import Parallel, delayed

fetcher = DocumentFetcher()
doc_tracker = DocumentTracker()
qdrant_manager = QdrantManager() 

# Calculate optimal number of workers (75% of CPU cores)
max_workers = max(1, int(os.cpu_count() * 0.75))
logger.info(f"Using {max_workers} parallel workers (75% of {os.cpu_count()} cores)")

In [ ]:
import asyncio
from typing import Any


# Wrapper function to process a single PDF using joblib
def process_pdf_wrapper(pdf_stream: Any, filename: str) -> tuple[list[Document], dict]:
    """process_document synchronously to use with joblib.

    Returns:
        Tuple of (documents, diagnostics)
    """
    diagnostics = {
        "filename": filename,
        "metadata_initialized": False,
        "stock_data_initialized": False,
        "error": None,
        "sample_metadata": None,
    }

    try:
        # CRITICAL: Initialize metadata and stock data in THIS worker process
        # The main process's data is NOT shared with worker processes
        from utils.data_helpers import (
            initialize_metadata_data,
            initialize_stock_data,
            is_metadata_initialized,
            is_stock_data_initialized,
        )

        # Create event loop for async initialization
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)

        try:
            # Initialize data in this worker process
            loop.run_until_complete(initialize_stock_data())
            loop.run_until_complete(initialize_metadata_data())

            diagnostics["metadata_initialized"] = is_metadata_initialized()
            diagnostics["stock_data_initialized"] = is_stock_data_initialized()

            # Process the PDF
            result = loop.run_until_complete(process_document(pdf_stream, file_type="pdf"))

            # Capture sample metadata from first document
            if result and len(result) > 0:
                diagnostics["sample_metadata"] = result[0].metadata

            return result, diagnostics
        finally:
            loop.close()

    except Exception as e:
        diagnostics["error"] = str(e)
        import traceback

        diagnostics["traceback"] = traceback.format_exc()
        return [], diagnostics


async def process_fincode(fincode: str):
    """Process a single fincode: fetch, check existence, process, and embed documents."""
    logger.info(f"Starting processing for fincode: {fincode}")

    # fetch documents for the fincode
    try:
        docs = await fetcher.get_available_documents(
            fincode=fincode,
        )
        for doc in docs:
            logger.info(
                f"Document: {doc.filename} ({doc.category}) - {doc.document_date}"
            )
    except Exception as e:
        logger.error(f"Error fetching documents for fincode {fincode}: {e}")
        return

    try:
        # check which documents already exist in vector store
        exists = await doc_tracker.check_documents_exist([doc.filename for doc in docs])
    except Exception as e:
        logger.error(f"Error checking document existence for fincode {fincode}: {e}")
        return

    # Collect PDFs that need processing
    try:
        pdfs_to_process = []

        for doc in docs:
            if not exists[doc.filename]:
                try:
                    logger.info(
                        f"Document {doc.filename} does not exist in vector store. Proceeding to fetch."
                    )
                    pdf = await fetcher.get_pdf_doc_stream(doc.filename, doc.category)
                    file_size = len(pdf.stream.getbuffer()) / 1024 / 1024
                    logger.info(f"Document size: {file_size:.2f} MB - {doc.filename}")

                    pdfs_to_process.append((pdf, doc.filename))
                except Exception as e:
                    logger.error(f"Error fetching document {doc.filename}: {e}")
                    continue
    except Exception as e:
        logger.error(f"Error collecting PDFs for fincode {fincode}: {e}")
        return

    # Process all PDFs in parallel using joblib (CPU-intensive operation)
    try:
        processed_docs: list[Document] = []

        if pdfs_to_process:
            logger.info(
                f"Processing {len(pdfs_to_process)} PDFs in parallel for fincode {fincode}"
            )

            # Use joblib to process PDFs in parallel
            results = Parallel(n_jobs=max_workers, backend="loky", verbose=10)(
                delayed(process_pdf_wrapper)(pdf, filename)
                for pdf, filename in pdfs_to_process
            )

            # Process results and log diagnostics
            for docs_result, diag in results:
                logger.info(f"\n--- Diagnostics for {diag['filename']} ---")
                logger.info(f"Metadata initialized: {diag['metadata_initialized']}")
                logger.info(f"Stock data initialized: {diag['stock_data_initialized']}")
                if diag["error"]:
                    logger.error(f"Error: {diag['error']}")
                    logger.error(f"Traceback: {diag.get('traceback', 'N/A')}")
                if diag["sample_metadata"]:
                    logger.info(f"Sample metadata: {diag['sample_metadata']}")
                logger.info("---\n")

                if docs_result:
                    processed_docs.extend(docs_result)

            logger.info(
                f"Processed {len(processed_docs)} document chunks from {len(pdfs_to_process)} PDFs"
            )
    except Exception as e:
        logger.error(f"Error processing PDFs for fincode {fincode}: {e}")
        return

    # Select documents that need to be embedded
    try:
        docs_to_embed: list[Document] = []
        for doc in processed_docs:
            source = doc.metadata.get("source", "")
            if not exists.get(source, False):
                docs_to_embed.append(doc)

        logger.info(
            f"Found {len(docs_to_embed)} document chunks to embed for fincode {fincode}"
        )
    except Exception as e:
        logger.error(f"Error selecting documents to embed for fincode {fincode}: {e}")
        return

    # Embed documents
    try:
        if isinstance(docs_to_embed, list) and len(docs_to_embed) > 0:
            result = await qdrant_manager.embed_documents(docs_to_embed)
            logger.info(
                f"Fincode {fincode} - Embedded {len(docs_to_embed)} chunks - Cost: ${result['estimated_cost_usd']:.4f}"
            )
        else:
            logger.info(f"No new documents to embed for fincode {fincode}. Skipping.")
    except Exception as e:
        logger.error(f"Error embedding documents for fincode {fincode}: {e}")
        return

    logger.info(f"Completed processing for fincode: {fincode}")

In [ ]:
# Process fincodes sequentially, but PDFs in parallel within each fincode
# This approach processes one fincode at a time, but uses all CPU cores for PDF processing
logger.info(f"Starting sequential processing of {len(fincodes)} fincodes...")
logger.info(f"PDFs within each fincode will be processed in parallel using {max_workers} workers")

for idx, fincode in enumerate(fincodes, 1):
    try:
        logger.info(f"[{idx}/{len(fincodes)}] Processing fincode: {fincode}")
        await process_fincode(fincode)
    except Exception as e:
        logger.error(f"Error processing fincode {fincode}: {e}")

logger.info("All fincodes processed!")